# DanmakuRender：按运行编号恢复处理与投稿

运行后先挂载 Drive，再填写历史 `run_id` 并从上到下运行。先校验 Drive 清单与文件，复用已完成成品及合并，缺失成品从原片和 ASS 渲染，再补做合并归档和多 P 投稿。原录制错误保留，无法补回未录到的内容。

开始前结束该编号的录制、补传及其他恢复实例。建议使用独立 GPU 运行时，不与正在录制的 Notebook 共用。投稿使用私有 Drive 的 `DMRColab/config/upload.yml` 和 `credentials/bilibili_upload.json`，不要在本 Notebook 粘贴 Cookie。`unknown` 只核对远端，无法确认时停止。

源码需包含 `colab_resume.py`；私有副本不会自动随仓库更新。结果单独保存到该运行目录的 `resume_manifest.json`，本 Notebook 保留运行时供检查。


In [ ]:
#@title ① 挂载私有 Drive（第一步）
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
RESUME_PHASE = 'drive'
RESUME_PARAMETERS = None
RESUME_COMMAND = None
print('Drive 已挂载；现在填写要恢复的 run_id。')


In [ ]:
#@title ② 选择历史任务
previous_phase = globals().get('RESUME_PHASE')
RESUME_PHASE = 'invalid'
RESUME_PARAMETERS = None
RESUME_COMMAND = None
RUN_ID = '' #@param {type:"string"}
UPLOAD = True #@param {type:"boolean"}
RECONCILE_BVID = '' #@param {type:"string"}
ENCODER = 'h264_nvenc' #@param ["h264_nvenc", "libx264"]
MEDIA_TIMEOUT_HOURS = 4 #@param {type:"number"}
MIN_FREE_GIB = 10 #@param {type:"number"}
import math, re
if previous_phase != 'drive':
    raise RuntimeError('请先运行第一步并成功挂载 Drive')
if not re.fullmatch(r'[0-9]{8}-[0-9]{6}-[0-9a-f]{8}', RUN_ID.strip()):
    raise ValueError('请填写完整运行编号，例如 20261011-224438-5faf4204')
if RECONCILE_BVID.strip() and not re.fullmatch(r'BV[0-9A-Za-z]{10}', RECONCILE_BVID.strip()):
    raise ValueError('候选 BVID 格式无效；通常留空')
if ENCODER not in ('h264_nvenc', 'libx264') or type(UPLOAD) is not bool:
    raise ValueError('参数无效')
if any(type(v) not in (int, float) or not math.isfinite(v) or v <= 0 for v in (MEDIA_TIMEOUT_HOURS, MIN_FREE_GIB)):
    raise ValueError('时限与磁盘阈值必须为有限正数')
RESUME_PARAMETERS = dict(run_id=RUN_ID.strip(), upload=UPLOAD, bvid=RECONCILE_BVID.strip(), encoder=ENCODER, timeout=MEDIA_TIMEOUT_HOURS * 3600, free=MIN_FREE_GIB)
BACKUP = DRIVE_ROOT / 'DMRColab/runs' / RESUME_PARAMETERS['run_id']
if not (BACKUP / 'manifest.json').is_file():
    raise RuntimeError('Drive 中未找到该编号的 manifest.json')
RESUME_PHASE = 'form'


In [ ]:
#@title ③ 下载本次恢复源码并安装依赖
previous_phase = globals().get('RESUME_PHASE')
RESUME_PHASE = 'invalid'
RESUME_COMMAND = None
if previous_phase != 'form':
    raise RuntimeError('请先填写并成功运行历史任务表单')
import subprocess, sys, tempfile
PROJECT = Path(tempfile.mkdtemp(prefix='dmr-recovery-', dir='/content')) / 'project'
def checked(command, **kwargs):
    return subprocess.run(command, check=True, **kwargs)
checked(['git', 'clone', '--depth', '1', '--branch', 'codex/colab-poc', 'https://github.com/zaixiakongyiji/DanmakuRenderForColab.git', str(PROJECT)])
if not (PROJECT / 'colab_resume.py').is_file():
    raise RuntimeError('仓库分支尚未发布恢复入口，请先提交并推送恢复代码')
print('源码 commit:', subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip())
checked(['apt-get', 'update', '-qq'])
checked(['apt-get', 'install', '-y', 'ffmpeg', 'fontconfig', 'fonts-noto-cjk'])
checked([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT / 'colab_support/requirements.txt')])
checked(['fc-cache', '-f'])
p = RESUME_PARAMETERS
RESUME_COMMAND = (sys.executable, str(PROJECT / 'colab_resume.py'), '--run-id', p['run_id'], '--drive-root', str(DRIVE_ROOT), '--encoder', p['encoder'], '--media-timeout', str(p['timeout']), '--min-free-gib', str(p['free']))
if not p['upload']:
    RESUME_COMMAND += ('--no-upload',)
if p['bvid']:
    RESUME_COMMAND += ('--reconcile-bvid', p['bvid'])
RESUME_PHASE = 'dependencies'


In [ ]:
#@title ④ 校验备份并显示恢复步骤
previous_phase = globals().get('RESUME_PHASE')
RESUME_PHASE = 'invalid'
if previous_phase != 'dependencies' or globals().get('RESUME_COMMAND') is None:
    raise RuntimeError('请先成功完成源码和依赖准备')
checked([*RESUME_COMMAND, '--inspect-only'], cwd=PROJECT)
RESUME_PHASE = 'inspected'


In [ ]:
#@title ⑤ 继续处理与投稿
previous_phase = globals().get('RESUME_PHASE')
RESUME_PHASE = 'invalid'
if previous_phase != 'inspected' or globals().get('RESUME_COMMAND') is None:
    raise RuntimeError('请先成功完成备份校验')
RESUME_PHASE = 'running'
try:
    result = subprocess.run(list(RESUME_COMMAND), cwd=PROJECT, check=False)
    print('恢复退出码:', result.returncode)
finally:
    RESUME_PHASE = 'finished'
    import json
    summary = BACKUP / 'resume_manifest.json'
    if summary.is_file():
        state = json.loads(summary.read_text(encoding='utf-8'))
        print(json.dumps({k: state.get(k) for k in ('run_id', 'status', 'phase', 'original_status', 'original_errors', 'errors', 'merge', 'upload')}, ensure_ascii=False, indent=2))
    print('结果目录:', BACKUP)
    print('运行时保留。检查结果后请自行断开并删除运行时；重试请重新运行表单及后续步骤。')
